# Benchmark of LLM agent performance in uncertain environments
This notebook analyzes the performance of an llm trying to navigate an uncertain environment. Using a lookahead search it decides which steps are the best possible moves. To value each step another llm is leveraged with a list of hypothese and policies about the environment.


In [1]:
# Setup + Imports
from evaluation.Benchmark import Benchmark
from openai import OpenAI
import gymnasium as gym
from tinydb import TinyDB
import os
from navigation.environments.FrozenLakeEnv import FrozenLakeEnv
from navigation.environments.FrozenLakeShadowEnv import FrozenLakeShadowEnv
from optimization.prompts.FrozenLakePrompts import FrozenLakePrompts

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)
model = "openai/gpt-oss-120b"

## FrozenLake Benchmarks

In [2]:
policyDb = TinyDB("../store/policies.json")
hypothesesDb = TinyDB("../store/hypotheses.json")

env = FrozenLakeEnv(env = gym.make("FrozenLake-v1", render_mode="ansi", desc=None, map_name="4x4", is_slippery=False, success_rate=0.7, reward_schedule=(1, 0, 0)))
shadow_env = FrozenLakeShadowEnv(hypothesesDb=hypothesesDb, policyDb=policyDb, client=client, model=model)
optimizationPrompts = FrozenLakePrompts(policyDb=policyDb, hypothesesDb=hypothesesDb)

bench_smallMap_noLlm = Benchmark(env=env, shadow_env=shadow_env, policyDb=policyDb, hypothesesDb=hypothesesDb, optimizationPrompts=optimizationPrompts, client=client, model=model)
policyDb.truncate()
hypothesesDb.truncate()

In [3]:
# no lookahead, no llm action, small map
metrics_nlook_nllm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=30, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

Agent Response: ChatCompletionMessage(content='{"move": "move_right", "value": 78}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="We need to evaluate moves. No environment hypotheses or policies given (empty). So just consider distance to goal and hazards H? Grid shows S top-left? Let's parse.\n\nCurrent state grid representation: 4 columns, 4 rows.\n\nRow1: [S] F  F  F \nRow2: F  H  F  H \nRow3: F  F  F  H \nRow4: H  F  F  G \n\nCoordinates: let's define (row, col) starting 0. S at (0,0). Goal G at (3,3). Hazards H at positions: (1,1), (1,3), (2,3), (3,0). Free cells F elsewhere.\n\nAvailable moves from S: can move right to (0,1), down to (1,0). left/up would be out of bounds (assuming cannot). So move_left or move_up would be invalid or would stay? But we still need to rate them low. We must pick best move.\n\nCompute Manhattan distance to goal: S distance = |3-0|+|3-0| =6.\n\nMove right to (0,1): distance = |3-0|+|3-1|

In [4]:
# no lookahead, llm action, small map
metrics_nlook_llm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=1, lookahead_depth=1, max_nav_steps=30, use_llm_action=True, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

Agent Response: ChatCompletionMessage(content='{"move": "move_down", "value": 80}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to output best move with value. Let\'s parse grid.\n\nCurrent state representation: lines:\n\n"[S] F  F  F "\n" F  H  F  H "\n" F  F  F  H "\n" H  F  F  G "\n\nWe need positions coordinates. Let\'s assume grid 4x4, rows top to bottom. S at (0,0). Goal G at (3,3). Hazards H at various cells.\n\nList Hazards: row1 col? Actually line2: " F  H  F  H " so at (1,1) and (1,3). Row3: " F  F  F  H " hazard at (2,3). Row4: " H  F  F  G " hazard at (3,0). So hazards at (1,1), (1,3), (2,3), (3,0). Start (0,0). Goal (3,3).\n\nAvailable moves from S: left (out of bounds), right -> (0,1) which is F (free). up (out of bounds), down -> (1,0) which is F (free). So left/up invalid maybe considered dangerous. We must rate.\n\nGoal is southeast direction. Move down is towards goal (increase row). Move right 

In [5]:
# lookahead, no llm action, small map
metrics_2look_nllm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=30, use_llm_action=False, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

Agent Response: ChatCompletionMessage(content='{"move": "move_right", "value": 70}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="The task: we have to rate moves left/right/up/down from current position. The environment grid is shown:\n\nCurrent State:\n```\n[S] F  F  F \n F  H  F  H \n F  F  F  H \n H  F  F  G \n```\n\nInterpretation: Row1: [S] F F F\nRow2: F H F H\nRow3: F F F H\nRow4: H F F G\n\nWe need coordinates. Let's assume grid indexes: rows from top 0 to 3, columns left to right 0 to 3. S at (0,0). Goal G at (3,3). H maybe hazard (danger). F is free.\n\nAvailable moves: left, right, up, down. From (0,0), moving left would go out of bounds (maybe invalid). Up also out of bounds. Right would move to (0,1) which is F. Down would move to (1,0) which is F.\n\nWe must consider policies/hypotheses. The prompt includes sections but empty. So no explicit policies/hypotheses. So just evaluate based on distance to goal.\n\

In [6]:
# lookahead, llm action, small map
metrics_2look_llm_small = bench_smallMap_noLlm.run(name="frozenlake_small", iteration_depth=10, lookahead_sample_size=2, lookahead_depth=2, max_nav_steps=30, use_llm_action=True, print_debug=True)
policyDb.truncate()
hypothesesDb.truncate()

Agent Response: ChatCompletionMessage(content='{"move": "move_right", "value": 78}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="We need to parse the problem. The environment grid:\n\nCurrent State shows:\n\nRow1: [S] F  F  F \nRow2:  F  H  F  H \nRow3:  F  F  F  H \nRow4:  H  F  F  G \n\nSo a 4x4 grid. S at (row1,col1). Goal G at (row4,col4). H are hazards (dangerous). F are free.\n\nAvailable moves: left, right, up, down. From S at top-left corner (row1, col1). Moves:\n\n- move_left: would go out of bounds (col0) - likely invalid. Might be considered dangerous or impossible.\n\n- move_up: out of bounds (row0) - invalid.\n\n- move_right: to (row1,col2) which is F.\n\n- move_down: to (row2,col1) which is F.\n\nWe need to rate each.\n\nWe have no explicit policies/hypotheses given (the sections are empty). So base on general: moving towards goal is good; avoid hazards. Both right and down move towards goal (down moves clo